In [3]:
from __future__ import annotations
import requests


topicos = [
    'Aposentadoria e Pensão',
		'Categoria Profissional Especial',
		'Contrato Individual de Trabalho',
		'Direito Coletivo do Trabalho',
		'Direito de Greve / Lockout',
		'Direito Individual do Trabalho',
		'Direito Sindical e Questões Análogas',
		'Duração do Trabalho',
		'Férias',
		'Outras Relações de Trabalho',
		'Prescrição',
		'Prescrição e Decadência no Direito do Trabalho',
		'Questões de Alta Complexidade, Grande Impacto e Repercussão',
		'Redução à Condição Análoga à de Escravo',
		'Rescisão do Contrato de Trabalho',
		'Rescisão do Contrato de Trabalho',
		'Responsabilidade Civil do Empregador',
		'Responsabilidade Solidária / Subsidiária',
		'Sentença Normativa',
		'Verbas Remuneratórias, Indenizatórias e Benefícios'
]

query_ = {
  "size": 100,
  "query": {
    "bool": {
      "must": [
        { "match": { "assuntos.nome": nome} } for nome in topicos
      ]
    }
  }
}


query = {
  "size": 10,
  "query": {
    "bool": {
      "must": [
        { "match": { "assuntos.nome": "Transporte Aéreo" } },
        { "match": { "assuntos.nome": "Indenização por Dano Moral" } }
      ]
    }
  }
}

url = "https://api-publica.datajud.cnj.jus.br/api_publica_trt6/_search"

headers = {
    "Authorization": "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="
}

In [ ]:
from shared.schemas import ProcessoResumido, ProcessoResumidoResponse, SearchResponse


def mapear_processos(search_response: SearchResponse) -> ProcessoResumidoResponse:
    processos = []
    for hit in search_response.hits.hits:
        processos.append(
            ProcessoResumido(
                numeroProcesso=hit.source.numeroProcesso,
                classe=hit.source.classe.nome,
                sistema=hit.source.sistema.nome,
                formato=hit.source.formato.nome,
                tribunal=hit.source.tribunal,
                dataHoraUltimaAtualizacao=hit.source.dataHoraUltimaAtualizacao,
                grau="2" if hit.source.grau == "G2" else "1",
                timestamp=hit.source.timestamp,
                dataAjuizamento=hit.source.dataAjuizamento,
                id=hit.source.id,
                orgaoJulgador=hit.source.orgaoJulgador.nome,
                assuntos=list(map(lambda a: a.nome, hit.source.assuntos))
            )
        )

    return ProcessoResumidoResponse(processos=processos)



In [5]:
import json

def fetch():
    response = requests.get(url, headers=headers, json=query, timeout=30)
    response.raise_for_status()
    return response.json()

def fetch_or_load():
    try:
        return fetch()
    except (requests.RequestException, ValueError) as e:
        print(str(e))
        with open('search_response.json', encoding='utf-8') as f:
            return json.load(f)

response = fetch()
print(len(response))

4


In [6]:

search_response = SearchResponse.model_validate(response)
processos = mapear_processos(search_response)

with open("search_response.json", "w", encoding='utf-8') as f:
    f.write(processos.model_dump_json(indent=2, ensure_ascii=False))
